In [1]:
# install libraries
import os
import json
import numpy as np
import pandas as pd

from datetime import datetime
from pathlib import Path
from collections import defaultdict

In [4]:
## Load NY-Dryden GeoJSON
current_path = Path(os.getcwd())
data_path = current_path / '..'/ 'data' 
geojson_path = data_path / 'Main Street Dryden.geojson'
ami_folder = current_path / '..'/ 'NREL'

ind_load_csv = ami_folder / 'consumption'/ 'ch1_4301002_individual.csv'
location_ref = ami_folder / 'location_reference.csv'
agg_load_csv = ami_folder / 'consumption'/ 'ch1_4301002_aggregated.csv'

print('done')

done


In [5]:
# read geojson

# Load GeoJSON file
with open(geojson_path, "r") as f:
    geojson = json.load(f)

# Loop through features
for feature in geojson.get("features", []):
    geometry = feature.get("geometry", {})
    properties = feature.get("properties", {})
    addr_housenumber = properties.get("addr:housenumber")
    addr_street = properties.get("addr:street", "").lower()
    building = properties.get("building", "").lower()

    print("House Number:", addr_housenumber)
    print("Street:", addr_street)
    print("Building:", building)
    #print("Geometry:", geometry.get("type"), geometry.get("coordinates"))
    print("---")

House Number: 36
Street: union street
Building: school
---
House Number: 2
Street: east main street
Building: yes
---
House Number: 36
Street: west main street
Building: yes
---
House Number: 22
Street: west main street
Building: yes
---
House Number: 8-10
Street: west main street
Building: yes
---
House Number: 14-20
Street: west main street
Building: yes
---
House Number: 7
Street: library street
Building: yes
---
House Number: 6
Street: lewis street
Building: yes
---
House Number: 50
Street: west main street
Building: yes
---
House Number: 44
Street: west main street
Building: yes
---
House Number: None
Street: 
Building: yes
---
House Number: 30-34
Street: west main street
Building: roof
---
House Number: 5
Street: library street
Building: yes
---
House Number: None
Street: 
Building: yes
---
House Number: 62
Street: west main street
Building: yes
---
House Number: 68
Street: west main street
Building: yes
---
House Number: 4
Street: wall street
Building: yes
---
House Number: 13
S

In [11]:
## combine location_ref and ind load csv

df_loc_ref = pd.read_csv(location_ref)
df_ind_load_df = pd.read_csv(ind_load_csv)


# for sp_id in df_loc_ref['servicepointid'].unique():
#     sp_data = df_loc_ref[df_loc_ref['servicepointid'] == sp_id]
    
#     lat = sp_data['latitude']
#     lon = sp_data['longitude']
#     node_id = sp_data['xfr_gisid']
#     ami = sp_data['amimeter']
#     res = sp_data['Residential']

# Standardize servicepointid column names
df_loc_ref['SERVICEPOINTID'] = df_loc_ref['servicepointid']
df_loc_ref.drop(columns=['servicepointid'], inplace=True)

# Merge on SERVICEPOINTID
combined_df = df_ind_load_df.merge(df_loc_ref, on='SERVICEPOINTID', how='left')

combined_df.to_csv("ami_location_combined.csv", index=False)
print(combined_df.head())
print('done')



   SERVICEPOINTID  CHANNELNUMBER          ENDTIME_EST  INTERVAL_READ  circuit  \
0      6000973858              1  2023-06-13 14:00:00          0.057  4301002   
1      6000973858              1  2023-06-13 14:15:00          0.051  4301002   
2      6000973858              1  2023-06-13 14:30:00          0.013  4301002   
3      6000973858              1  2023-06-13 14:45:00          0.031  4301002   
4      6000973858              1  2023-06-13 15:00:00          0.037  4301002   

     xfr_gisid amimeter Residential  latitude  longitude  
0  220270218.0        Y         Res  42.49872 -76.272127  
1  220270218.0        Y         Res  42.49872 -76.272127  
2  220270218.0        Y         Res  42.49872 -76.272127  
3  220270218.0        Y         Res  42.49872 -76.272127  
4  220270218.0        Y         Res  42.49872 -76.272127  
done


In [ ]:
## ATTACH SERVICEPOINT ID TO GEOJSON

import pandas as pd
import json
from shapely.geometry import shape, Point
from shapely.ops import nearest_points
from pathlib import Path

# Load GeoJSON
geojson_path = Path("/mnt/data/Main Street Dryden.geojson")
with open(geojson_path) as f:
    geojson_data = json.load(f)

# Load Excel with S.No. and coordinates
excel_path = Path("/mnt/data/location_reference_sno.xlsx")
location_df = pd.read_excel(excel_path)

# Prepare GeoJSON features
features = geojson_data["features"]
for feature in features:
    feature["geometry_obj"] = shape(feature["geometry"])
    feature["properties"]["Matched_SNo"] = None
    feature["properties"]["Min_Distance"] = float("inf")

# Assign each S.No. to the closest footprint only once
for _, row in location_df.iterrows():
    sno = row["S.No."]
    point = Point(row["longitude"], row["latitude"])

    min_dist = float("inf")
    best_match = None

    for feature in features:
        polygon = feature["geometry_obj"]
        nearest_geom = nearest_points(point, polygon)[1]
        dist = point.distance(nearest_geom)

        if dist < min_dist:
            min_dist = dist
            best_match = feature

    if best_match and min_dist < best_match["properties"]["Min_Distance"]:
        best_match["properties"]["Matched_SNo"] = str(sno)
        best_match["properties"]["Min_Distance"] = min_dist

# Cleanup geometry and distance helpers
for feature in features:
    del feature["geometry_obj"]
    if "Min_Distance" in feature["properties"]:
        del feature["properties"]["Min_Distance"]

# Save updated GeoJSON
geojson_data["features"] = features
output_geojson_path = "/mnt/data/Unique_Matched_Footprints.geojson"
with open(output_geojson_path, "w") as f:
    json.dump(geojson_data, f)


#####
#Out of a total of 792 S.No. entries:

# 164 S.No. values were successfully matched to a unique building footprint.

# 628 S.No. values remain unmatched.



In [ ]:
# VISUALIZE WHAT WAS NOT MATCHED

# Reload location reference data
location_df = pd.read_excel("/mnt/data/location_reference_sno.xlsx")

# Extract matched S.No. values from the cleaned GeoJSON
matched_snos = set()
for feature in geojson_data["features"]:
    matched = feature["properties"].get("Matched_SNo")
    if matched:
        matched_snos.add(int(matched))

# Separate matched and unmatched DataFrames
matched_df = location_df[location_df["S.No."].isin(matched_snos)]
unmatched_df = location_df[~location_df["S.No."].isin(matched_snos)]

# Plot using raw matplotlib
fig, ax = plt.subplots(figsize=(10, 10))

# Draw building footprints
for geom in geometry_list:
    if geom.geom_type == "Polygon":
        x, y = geom.exterior.xy
        ax.plot(x, y, color='gray', linewidth=1)
    elif geom.geom_type == "MultiPolygon":
        for part in geom.geoms:
            x, y = part.exterior.xy
            ax.plot(x, y, color='gray', linewidth=1)

# Plot matched and unmatched points
ax.scatter(matched_df["longitude"], matched_df["latitude"], c='green', label="Matched S.No.", s=10)
ax.scatter(unmatched_df["longitude"], unmatched_df["latitude"], c='red', label="Unmatched S.No.", s=10)
ax.set_title("S.No. Coordinates vs Building Footprints")
ax.legend()
plt.tight_layout()
plt.show()
